# Streamflow USGS comparisons 

In [2]:
# ==== CELL 1: Data Retrieval ====
# Read site IDs from text file and retrieve USGS discharge measurements
from dataretrieval import nwis
from IPython.display import display
import pandas as pd
from pathlib import Path

def read_site_ids(file_path):
    """
    Read site IDs from a text file and clean them up.
    Removes 'USGS' prefix and any whitespace.
    """
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    # Clean up the site IDs: remove 'USGS', strip whitespace, filter empty lines
    site_ids = []
    for line in lines:
        line = line.strip()  # Remove leading/trailing whitespace
        if line:  # Skip empty lines
            # Remove 'USGS' prefix if present (case insensitive)
            if line.upper().startswith('USGS '):
                site_id = line[5:]  # Remove 'USGS ' (5 characters)
            else:
                site_id = line
            site_ids.append(site_id.strip())
    
    return site_ids

# Path to your text file with site IDs
site_file_path = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\original_data\stream_sb_usgs.txt"

try:
    # Read and clean site IDs
    sites = read_site_ids(site_file_path)
    print(f"Found {len(sites)} site IDs:")
    print(sites[:10])  # Show first 10 sites as preview
    if len(sites) > 10:
        print(f"... and {len(sites) - 10} more sites")
    
    # Retrieve discharge measurements from all sites
    print(f"\nRetrieving discharge measurements from {len(sites)} USGS sites...")
    sb_streams = nwis.get_discharge_measurements(
        sites=sites, 
        start="1800-01-01", 
        end="1979-12-31"
    )
    
    print("Retrieved " + str(len(sb_streams[0])) + " data values.")
    print(f"Data columns: {sb_streams[0].columns.tolist()}")  # Show column names for verification
    display(sb_streams[0].head())  # Show first few rows instead of entire dataframe
    
    # Create output directory if it doesn't exist
    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    
    # Save to CSV
    csv_path = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\output\sb_streamflow_usgs_1800_1979.csv"
    sb_streams[0].to_csv(csv_path, index=False)
    print(f"Saved {len(sb_streams[0]):,} streamflow measurements to: {csv_path}")
    
    # Verify the file was written successfully
    if Path(csv_path).exists():
        file_size = Path(csv_path).stat().st_size
        print(f"File successfully written. Size: {file_size:,} bytes")
    else:
        print("WARNING: CSV file was not created successfully")
    
except FileNotFoundError:
    print(f"Error: Could not find the file '{site_file_path}'")
    print("Please update the 'site_file_path' variable with the correct path to your text file.")
except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()

print("=== DATA RETRIEVAL COMPLETE ===")
print("You can now run the processing cell...")

# ==== CELL 2: Data Processing ====
print("\n=== PROCESSING STREAMFLOW NWIS DATA ===")

import pandas as pd
from pathlib import Path
import numpy as np

# Load the streamflow CSV data
csv_path = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\output\sb_streamflow_usgs_1800_1979.csv"

try:
    # Check if file exists before trying to read it
    if not Path(csv_path).exists():
        raise FileNotFoundError(f"CSV file not found at {csv_path}")
    
    # Load the streamflow measurements
    sb_streamflow = pd.read_csv(csv_path)
    print(f"Loaded {len(sb_streamflow):,} streamflow measurements")
    
    # Display basic info about the data
    print(f"Columns: {sb_streamflow.columns.tolist()}")
    
    # Check if measurement_dt column exists
    if 'measurement_dt' not in sb_streamflow.columns:
        print("Available columns:", sb_streamflow.columns.tolist())
        raise KeyError("Column 'measurement_dt' not found in the data. Please check column names.")
    
    print(f"Date range in measurement_dt: {sb_streamflow['measurement_dt'].min()} to {sb_streamflow['measurement_dt'].max()}")
    
    # Convert measurement_dt to datetime and extract year
    # Handle mixed date formats: M/D/YYYY and M/D/YYYY HH:MM
    print("Converting dates to datetime format...")
    
    # Custom function to handle the specific date formats
    def parse_mixed_dates(date_str):
        if pd.isna(date_str):
            return pd.NaT
        
        date_str = str(date_str).strip()
        
        # Try format 1: M/D/YYYY HH:MM (e.g., "2/19/1942 23:00")
        try:
            return pd.to_datetime(date_str, format='%m/%d/%Y %H:%M')
        except:
            pass
        
        # Try format 2: M/D/YYYY (e.g., "1/14/1942")
        try:
            return pd.to_datetime(date_str, format='%m/%d/%Y')
        except:
            pass
        
        # Try pandas default parsing as fallback
        try:
            return pd.to_datetime(date_str)
        except:
            return pd.NaT
    
    # Apply the custom parsing function
    sb_streamflow['measurement_dt'] = sb_streamflow['measurement_dt'].apply(parse_mixed_dates)
    sb_streamflow['year'] = sb_streamflow['measurement_dt'].dt.year
    
    # Check for any failed conversions
    failed_conversions = sb_streamflow['measurement_dt'].isna().sum()
    if failed_conversions > 0:
        print(f"Warning: {failed_conversions} dates could not be parsed")
        # Show some examples of failed conversions
        failed_dates = sb_streamflow[sb_streamflow['measurement_dt'].isna()]['measurement_dt'].head()
        if not failed_dates.empty:
            print("Examples of failed date conversions:", failed_dates.tolist())
    else:
        print("All dates parsed successfully")
    
    # Remove rows with failed date conversions for the analysis
    sb_streamflow_clean = sb_streamflow.dropna(subset=['measurement_dt', 'year'])
    if len(sb_streamflow_clean) < len(sb_streamflow):
        print(f"Removed {len(sb_streamflow) - len(sb_streamflow_clean)} rows with invalid dates")
        sb_streamflow = sb_streamflow_clean
    
    print(f"Year range: {sb_streamflow['year'].min()} to {sb_streamflow['year'].max()}")
    print(f"Unique sites: {sb_streamflow['site_no'].nunique()}")
    
    # ==== Summary 1: Each unique site ID with start year, end year, and total measurements ====
    print("\n=== CREATING SITE SUMMARY (Start Year, End Year, and Total Measurements) ===")
    
    site_summary = sb_streamflow.groupby('site_no').agg({
        'year': ['min', 'max'],  # First/earliest year and last/latest year
        'measurement_dt': 'count'  # Total number of measurements
    }).reset_index()
    
    # Flatten the multi-level column names
    site_summary.columns = ['site_no', 'start_year', 'end_year', 'total_measurements']
    
    # Calculate the span of years for each site
    site_summary['year_span'] = site_summary['end_year'] - site_summary['start_year'] + 1
    
    # Sort by start year then site number
    site_summary = site_summary.sort_values(['start_year', 'site_no'])
    
    print(f"Created summary for {len(site_summary):,} unique sites")
    print("Sample of site summary with start and end years:")
    print(site_summary.head())
    
    # Save summary 1
    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    
    summary1_path = output_dir / "sb_streamflow_sites_start_end_years.csv"
    site_summary.to_csv(summary1_path, index=False)
    print(f"Saved site summary with start/end years: {summary1_path}")
    
    # ==== Summary 2: Each unique site ID and each unique year it has measurements ====
    print("\n=== CREATING SITE-YEAR SUMMARY (All Years) ===")
    
    # Get unique combinations of site_no and year, with measurement count per year
    site_year_summary = sb_streamflow.groupby(['site_no', 'year']).agg({
        'measurement_dt': 'count'  # Number of measurements in this year
    }).reset_index()
    
    # Rename column for clarity
    site_year_summary.columns = ['site_no', 'year', 'measurements_in_year']
    
    # Sort by site number then year
    site_year_summary = site_year_summary.sort_values(['site_no', 'year'])
    
    print(f"Created site-year summary with {len(site_year_summary):,} site-year combinations")
    print("Sample of site-year summary:")
    print(site_year_summary.head(10))
    
    # Save summary 2
    summary2_path = output_dir / "sb_streamflow_sites_all_years.csv"
    site_year_summary.to_csv(summary2_path, index=False)
    print(f"Saved site-year summary: {summary2_path}")
    
    # ==== Additional Analysis ====
    print("\n=== ADDITIONAL ANALYSIS ===")
    
    print("Site measurement statistics:")
    measurement_stats = site_summary['total_measurements'].describe()
    print(measurement_stats)
    
    print(f"\nYear span statistics:")
    span_stats = site_summary['year_span'].describe()
    print(span_stats)
    
    print(f"\nDecade distribution of start years:")
    site_summary['start_decade'] = (site_summary['start_year'] // 10) * 10
    decade_counts = site_summary['start_decade'].value_counts().sort_index()
    print(decade_counts)
    
    print(f"\nSites with most measurements:")
    top_sites = site_summary.nlargest(5, 'total_measurements')[['site_no', 'start_year', 'end_year', 'total_measurements', 'year_span']]
    print(top_sites)
    
    print(f"\nEarliest starting sites:")
    earliest_sites = site_summary.nsmallest(5, 'start_year')[['site_no', 'start_year', 'end_year', 'total_measurements', 'year_span']]
    print(earliest_sites)
    
    print(f"\nSites with longest monitoring periods:")
    longest_span = site_summary.nlargest(5, 'year_span')[['site_no', 'start_year', 'end_year', 'total_measurements', 'year_span']]
    print(longest_span)
    
    # ==== Create a visualization-ready summary for the decadal analysis ====
    print("\n=== CREATING DECADE SUMMARY FOR MAPPING ===")
    
    # Add decade to site summary for mapping
    site_mapping_summary = site_summary.copy()
    site_mapping_summary['decade'] = (site_mapping_summary['start_year'] // 10) * 10
    
    # For mapping, we'll need coordinates - but those will come from the site metadata
    # This summary can be merged with site location data later
    
    summary3_path = output_dir / "sb_streamflow_sites_for_mapping.csv"
    site_mapping_summary.to_csv(summary3_path, index=False)
    print(f"Saved mapping summary: {summary3_path}")
    
    print(f"\n=== SUMMARY FILES CREATED ===")
    print(f"1. Site start/end years: {summary1_path}")
    print(f"2. Site-year combinations: {summary2_path}")
    print(f"3. Mapping ready: {summary3_path}")

except FileNotFoundError:
    print(f"ERROR: Could not find the streamflow CSV file at: {csv_path}")
    print("Make sure you've run the data retrieval cell first and that it completed successfully.")
    print("Check that the CSV file was actually created in the expected location.")
except KeyError as e:
    print(f"ERROR: Missing expected column in the data: {e}")
    print("The data structure may have changed. Check the column names in your CSV file.")
except Exception as e:
    print(f"ERROR processing data: {e}")
    print("Check that your CSV file has the expected columns (site_no, measurement_dt, etc.)")
    import traceback
    traceback.print_exc()

Found 83 site IDs:
['11119500', '11119530', '11119660', '11119700', '11119745', '11119750', '11119760', '11119770', '11119780', '11119900']
... and 73 more sites

Retrieving discharge measurements from 83 USGS sites...
Retrieved 1948 data values.
Data columns: ['agency_cd', 'site_no', 'measurement_nu', 'measurement_dt', 'tz_cd', 'q_meas_used_fg', 'party_nm', 'site_visit_coll_agency_cd', 'gage_height_va', 'discharge_va', 'measured_rating_diff', 'gage_va_change', 'gage_va_time', 'control_type_cd', 'discharge_cd']


,agency_cd,site_no,measurement_nu,measurement_dt,tz_cd,q_meas_used_fg,party_nm,site_visit_coll_agency_cd,gage_height_va,discharge_va,measured_rating_diff,gage_va_change,gage_va_time,control_type_cd,discharge_cd
0,USGS,11119500,1,1940-12-31,NaN,Yes,UNK,USGS,2.02,0.25,Good,0.00,0.2,NaN,NONE
1,USGS,11119500,2,1941-01-07,NaN,Yes,UNK,USGS,2.49,13.40,Unspecified,0.00,0.4,NaN,NONE
2,USGS,11119500,3,1941-01-08,NaN,Yes,UNK,USGS,2.38,8.20,Good,0.00,0.3,NaN,NONE
3,USGS,11119500,4,1941-01-09,NaN,Yes,UNK,USGS,2.34,5.50,Unspecified,0.00,0.3,NaN,NONE
4,USGS,11119500,5,1941-01-22,NaN,Yes,UNK,USGS,2.39,8.70,Good,-0.01,0.3,NaN,NONE


Saved 1,948 streamflow measurements to: C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\output\sb_streamflow_usgs_1800_1979.csv
File successfully written. Size: 148,193 bytes
=== DATA RETRIEVAL COMPLETE ===
You can now run the processing cell...

=== PROCESSING STREAMFLOW NWIS DATA ===
Loaded 1,948 streamflow measurements
Columns: ['agency_cd', 'site_no', 'measurement_nu', 'measurement_dt', 'tz_cd', 'q_meas_used_fg', 'party_nm', 'site_visit_coll_agency_cd', 'gage_height_va', 'discharge_va', 'measured_rating_diff', 'gage_va_change', 'gage_va_time', 'control_type_cd', 'discharge_cd']
Date range in measurement_dt: 1940-12-31 to 1979-12-26
Converting dates to datetime format...
All dates parsed successfully
Year range: 1940 to 1979
Unique sites: 21

=== CREATING SITE SUMMARY (Start Year, End Year, and Total Measurements) ===
Created summary for 21 unique sites
Sample of site summary with start and end years:
     site_no  start_year  end_year  total_measurements  year_span
0 

In [2]:
# ==== Process Streamflow NWIS Data for Summaries ====
print("=== PROCESSING STREAMFLOW NWIS DATA ===")

import pandas as pd
from pathlib import Path
import numpy as np

# Load the streamflow CSV data
csv_path = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\output\sb_streamflow_usgs_1800_1979.csv"

try:
    # Load the streamflow measurements
    sb_streamflow = pd.read_csv(csv_path)
    print(f"Loaded {len(sb_streamflow):,} streamflow measurements")
    
    # Display basic info about the data
    print(f"Columns: {sb_streamflow.columns.tolist()}")
    print(f"Date range in measurement_dt: {sb_streamflow['measurement_dt'].min()} to {sb_streamflow['measurement_dt'].max()}")
    
    # Convert measurement_dt to datetime and extract year
    # Handle mixed date formats: M/D/YYYY and M/D/YYYY HH:MM:SS TZ
    sb_streamflow['measurement_dt'] = pd.to_datetime(sb_streamflow['measurement_dt'], errors='coerce')
    sb_streamflow['year'] = sb_streamflow['measurement_dt'].dt.year
    
    # Check for any failed conversions
    failed_conversions = sb_streamflow['measurement_dt'].isna().sum()
    if failed_conversions > 0:
        print(f"Warning: {failed_conversions} dates could not be parsed")
    else:
        print("All dates parsed successfully")
    
    print(f"Year range: {sb_streamflow['year'].min()} to {sb_streamflow['year'].max()}")
    print(f"Unique sites: {sb_streamflow['site_no'].nunique()}")
    
    # ==== Summary 1: Each unique site ID and its start year ====
    print("\n=== CREATING SITE SUMMARY (Start Year Only) ===")
    
    site_start_summary = sb_streamflow.groupby('site_no').agg({
        'year': 'min',  # First/earliest year
        'measurement_dt': 'count'  # Total number of measurements
    }).reset_index()
    
    # Rename columns for clarity
    site_start_summary.columns = ['site_no', 'start_year', 'total_measurements']
    
    # Sort by start year then site number
    site_start_summary = site_start_summary.sort_values(['start_year', 'site_no'])
    
    print(f"Created summary for {len(site_start_summary):,} unique sites")
    print("Sample of site start year summary:")
    print(site_start_summary.head())
    
    # Save summary 1
    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    
    summary1_path = output_dir / "sb_streamflow_sites_start_years.csv"
    site_start_summary.to_csv(summary1_path, index=False)
    print(f"Saved site start year summary: {summary1_path}")
    
    # ==== Summary 2: Each unique site ID and each unique year it has measurements ====
    print("\n=== CREATING SITE-YEAR SUMMARY (All Years) ===")
    
    # Get unique combinations of site_no and year, with measurement count per year
    site_year_summary = sb_streamflow.groupby(['site_no', 'year']).agg({
        'measurement_dt': 'count'  # Number of measurements in this year
    }).reset_index()
    
    # Rename column for clarity
    site_year_summary.columns = ['site_no', 'year', 'measurements_in_year']
    
    # Sort by site number then year
    site_year_summary = site_year_summary.sort_values(['site_no', 'year'])
    
    print(f"Created site-year summary with {len(site_year_summary):,} site-year combinations")
    print("Sample of site-year summary:")
    print(site_year_summary.head(10))
    
    # Save summary 2
    summary2_path = output_dir / "sb_streamflow_sites_all_years.csv"
    site_year_summary.to_csv(summary2_path, index=False)
    print(f"Saved site-year summary: {summary2_path}")
    
    # ==== Additional Analysis ====
    print("\n=== ADDITIONAL ANALYSIS ===")
    
    print("Site measurement statistics:")
    measurement_stats = site_start_summary['total_measurements'].describe()
    print(measurement_stats)
    
    print(f"\nDecade distribution of start years:")
    site_start_summary['start_decade'] = (site_start_summary['start_year'] // 10) * 10
    decade_counts = site_start_summary['start_decade'].value_counts().sort_index()
    print(decade_counts)
    
    print(f"\nSites with most measurements:")
    top_sites = site_start_summary.nlargest(5, 'total_measurements')
    print(top_sites)
    
    print(f"\nEarliest starting sites:")
    earliest_sites = site_start_summary.nsmallest(5, 'start_year')
    print(earliest_sites)
    
    # ==== Create a visualization-ready summary for the decadal analysis ====
    print("\n=== CREATING DECADE SUMMARY FOR MAPPING ===")
    
    # Add decade to site summary for mapping
    site_mapping_summary = site_start_summary.copy()
    site_mapping_summary['decade'] = (site_mapping_summary['start_year'] // 10) * 10
    
    # For mapping, we'll need coordinates - but those will come from the site metadata
    # This summary can be merged with site location data later
    
    summary3_path = output_dir / "sb_streamflow_sites_for_mapping.csv"
    site_mapping_summary.to_csv(summary3_path, index=False)
    print(f"Saved mapping summary: {summary3_path}")
    
    print(f"\n=== SUMMARY FILES CREATED ===")
    print(f"1. Site start years: {summary1_path}")
    print(f"2. Site-year combinations: {summary2_path}")
    print(f"3. Mapping ready: {summary3_path}")

except FileNotFoundError:
    print(f"ERROR: Could not find the streamflow CSV file at: {csv_path}")
    print("Make sure you've saved your sb_streams data to this location first.")
except Exception as e:
    print(f"ERROR processing data: {e}")
    print("Check that your CSV file has the expected columns (site_no, measurement_dt, etc.)")

=== PROCESSING STREAMFLOW NWIS DATA ===
Loaded 1,948 streamflow measurements
Columns: ['agency_cd', 'site_no', 'measurement_nu', 'measurement_dt', 'tz_cd', 'q_meas_used_fg', 'party_nm', 'site_visit_coll_agency_cd', 'gage_height_va', 'discharge_va', 'measured_rating_diff', 'gage_va_change', 'gage_va_time', 'control_type_cd', 'discharge_cd']
Date range in measurement_dt: 1940-12-31 to 1979-12-26
Year range: 1940.0 to 1979.0
Unique sites: 21

=== CREATING SITE SUMMARY (Start Year Only) ===
Created summary for 21 unique sites
Sample of site start year summary:
     site_no  start_year  total_measurements
0   11119500      1940.0                 399
5   11120500      1941.0                 815
9   11123500      1952.0                  28
12  11132500      1952.0                  27
18  11138500      1952.0                  51
Saved site start year summary: output\sb_streamflow_sites_start_years.csv

=== CREATING SITE-YEAR SUMMARY (All Years) ===
Created site-year summary with 204 site-year 

## old code below ignore as of 9/3

In [ ]:
# ==== USGS Streamflow Analysis and Comparison ====
print("=== STREAMFLOW DATA ANALYSIS ===")

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import re
from pathlib import Path
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries loaded successfully")

# ==== Load Streamflow Data from CSV Files ====
print("\n=== LOADING STREAMFLOW DATA FROM CSV FILES ===")

# Define file paths - Updated to use your measurement data
nwis_measurements_csv = Path(r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\output\sb_streamflow_usgs_1800_1979.csv")
nwis_sites_csv = Path(r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\output\sb_streamflow_sites_for_mapping.csv")
restored_csv = Path(r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\plots\run_20250825_112711\santa_barbara_county\santa_barbara_sites.csv")

try:
    # Load NWIS measurement data (individual measurements)
    print("Loading NWIS streamflow measurements...")
    nwis_measurements = pd.read_csv(nwis_measurements_csv)
    print(f"Loaded NWIS measurements: {len(nwis_measurements):,} individual measurements")
    
    # Parse dates and extract year
    nwis_measurements['measurement_dt'] = pd.to_datetime(nwis_measurements['measurement_dt'], errors='coerce')
    nwis_measurements['year'] = nwis_measurements['measurement_dt'].dt.year
    nwis_measurements['decade'] = (nwis_measurements['year'] // 10) * 10
    
    # Load NWIS site summary (unique sites)
    print("Loading NWIS site summary...")
    nwis_sites = pd.read_csv(nwis_sites_csv)
    print(f"Loaded NWIS sites: {len(nwis_sites):,} unique sites")
    
    # Load restored historical data
    print("Loading restored USGS historical data...")
    restored_df = pd.read_csv(restored_csv)
    print(f"Loaded restored data: {len(restored_df):,} sites")
    
    # Filter to stream discharge sites only
    historical_streams = restored_df[restored_df['water_type_clean'] == 'Stream discharge'].copy()
    print(f"Historical stream discharge sites: {len(historical_streams):,}")
    
    # Add decade column for historical data
    if 'start_year' in historical_streams.columns:
        historical_streams['decade'] = (historical_streams['start_year'] // 10) * 10
    elif 'year_start' in historical_streams.columns:
        historical_streams['decade'] = (historical_streams['year_start'] // 10) * 10
    else:
        print("Warning: No start year column found in historical data")
        historical_streams['decade'] = 1940
    
    print(f"Data loaded successfully")
    
    # We need coordinates for NWIS sites - get them from NWIS web services
    print("\nGetting coordinates for NWIS sites...")
    from dataretrieval import nwis as nwis_service
    
    # Get unique site numbers
    unique_sites = nwis_sites['site_no'].unique()
    print(f"Getting coordinates for {len(unique_sites)} NWIS sites...")
    
    # Get site information including coordinates
    site_info = nwis_service.get_info(sites=unique_sites.tolist())
    site_coords = site_info[0][['site_no', 'dec_lat_va', 'dec_long_va']].copy()
    site_coords = site_coords.rename(columns={'dec_lat_va': 'latitude', 'dec_long_va': 'longitude'})
    
    # Convert site_no to string in ALL DataFrames BEFORE merging
    print("Converting site_no columns to consistent data types...")
    nwis_sites['site_no'] = nwis_sites['site_no'].astype(str)
    site_coords['site_no'] = site_coords['site_no'].astype(str)
    nwis_measurements['site_no'] = nwis_measurements['site_no'].astype(str)
    
    # NOW do the merges
    print("Merging coordinate data...")
    nwis_sites_with_coords = nwis_sites.merge(site_coords, on='site_no', how='left')
    nwis_measurements_with_coords = nwis_measurements.merge(site_coords, on='site_no', how='left')
    
    # Create GeoDataFrames for sites
    nwis_sites_gdf = gpd.GeoDataFrame(
        nwis_sites_with_coords,
        geometry=gpd.points_from_xy(nwis_sites_with_coords['longitude'], nwis_sites_with_coords['latitude']),
        crs="EPSG:4326"
    )
    nwis_sites_gdf['data_type'] = 'NWIS USGS'
    
    # Create GeoDataFrame for historical sites
    historical_sites_gdf = gpd.GeoDataFrame(
        historical_streams,
        geometry=gpd.points_from_xy(historical_streams['lon_combined'], historical_streams['lat_combined']),
        crs="EPSG:4326"
    )
    historical_sites_gdf['data_type'] = 'Restored USGS'
    
    # Create GeoDataFrames for measurements
    nwis_measurements_gdf = gpd.GeoDataFrame(
        nwis_measurements_with_coords,
        geometry=gpd.points_from_xy(nwis_measurements_with_coords['longitude'], nwis_measurements_with_coords['latitude']),
        crs="EPSG:4326"
    )
    nwis_measurements_gdf['data_type'] = 'NWIS USGS'
    
    print(f"Created GeoDataFrames successfully")
    print(f"  - NWIS sites: {len(nwis_sites_gdf):,}")
    print(f"  - NWIS measurements: {len(nwis_measurements_gdf):,}")
    print(f"  - Historical sites: {len(historical_sites_gdf):,}")
    
except Exception as e:
    print(f"ERROR: {e}")
    print("Check file paths and ensure dataretrieval package is installed")

# ==== Load Santa Barbara County Boundary ====
print("\n=== LOADING SANTA BARBARA COUNTY BOUNDARY ===")

sb_county_shapefile = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\sb_county_shapefile\sb_county_shp.shp"

try:
    sb_county = gpd.read_file(sb_county_shapefile)
    if sb_county.crs is None or sb_county.crs.to_string().upper() not in ("EPSG:4326", "WGS84"):
        sb_county = sb_county.to_crs("EPSG:4326")
    
    sb_county_geom = sb_county.union_all() if hasattr(sb_county, "union_all") else sb_county.unary_union
    SB_COUNTY_GDF = gpd.GeoSeries([sb_county_geom], crs="EPSG:4326")
    print("Loaded Santa Barbara County boundary")
    
except FileNotFoundError:
    print("Warning: County boundary file not found - will use data bounds")
    SB_COUNTY_GDF = None

# ==== PROPER EXTENT CALCULATION (Same as working script) ====
print("=== CALCULATING PROPER MAP EXTENT ===")

# Combine all data for extent calculation
all_mapping_data = []
if 'nwis_sites_gdf' in locals() and len(nwis_sites_gdf) > 0:
    all_mapping_data.append(nwis_sites_gdf)
if 'historical_sites_gdf' in locals() and len(historical_sites_gdf) > 0:
    all_mapping_data.append(historical_sites_gdf)

if all_mapping_data:
    # Combine all data to get proper bounds
    combined_data = gpd.GeoDataFrame(pd.concat(all_mapping_data, ignore_index=True))
    
    # Get bounds from the ACTUAL DATA (not the county shapefile)
    minx, miny, maxx, maxy = combined_data.total_bounds
    
    # Calculate proportional padding (same as working script)
    pad_lon = max(0.01, (maxx - minx) * 0.15)  # 15% padding
    pad_lat = max(0.01, (maxy - miny) * 0.15)
    
    # Set consistent extent variables
    SB_XMIN, SB_YMIN, SB_XMAX, SB_YMAX = (
        minx - pad_lon, 
        miny - pad_lat, 
        maxx + pad_lon, 
        maxy + pad_lat
    )
    
    print(f"Data extent: {minx:.3f}, {miny:.3f}, {maxx:.3f}, {maxy:.3f}")
    print(f"Map extent with padding: {SB_XMIN:.3f}, {SB_YMIN:.3f}, {SB_XMAX:.3f}, {SB_YMAX:.3f}")
else:
    print("Warning: No data found for extent calculation")
    # Fallback to county bounds if no data
    if 'sb_county' in locals():
        bounds = sb_county.total_bounds
        SB_XMIN, SB_YMIN, SB_XMAX, SB_YMAX = bounds
    else:
        # Default Santa Barbara area
        SB_XMIN, SB_YMIN, SB_XMAX, SB_YMAX = (-120.8, 34.2, -119.4, 34.9)

# Create output directory
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# ==== Map 1: Site-Based Comparison Map ====
print("\n=== CREATING SITE-BASED COMPARISON MAP ===")

def create_comparison_map(nwis_data, historical_data, title_suffix, filename):
    """Create a comparison map with proper extent"""
    
    # Calculate aspect ratio based on data bounds (not arbitrary)
    width = SB_XMAX - SB_XMIN
    height = SB_YMAX - SB_YMIN
    aspect_ratio = width / height
    fig, ax = plt.subplots(figsize=(10, 10/aspect_ratio), dpi=300)
    
    # Add California context
    try:
        ca_context_path = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\ca_shapefile\ca_poly.shp"
        if os.path.exists(ca_context_path):
            ca_context = gpd.read_file(ca_context_path).to_crs("EPSG:4326")
            ca_context_geom = ca_context.union_all() if hasattr(ca_context, "union_all") else ca_context.unary_union
            gpd.GeoSeries([ca_context_geom], crs="EPSG:4326").plot(
                ax=ax, color="#f0f0f0", edgecolor="#e0e0e0", linewidth=0.5, zorder=0
            )
    except Exception:
        pass
    
    # Plot county boundary
    if 'SB_COUNTY_GDF' in globals() and SB_COUNTY_GDF is not None:
        SB_COUNTY_GDF.plot(ax=ax, color="lightgray", edgecolor="black", linewidth=0.8, zorder=1)
    
    # Plot data
    nwis_data.plot(ax=ax, color='#4477AA', markersize=20, alpha=0.7, edgecolor='none', zorder=3, label='NWIS USGS Stream Data')
    historical_data.plot(ax=ax, color='#EE6677', markersize=20, alpha=0.7, edgecolor='none', zorder=2, label='Restored USGS Stream Data')
    
    # CONSISTENT extent setting (no arbitrary extensions)
    ax.set_xlim(SB_XMIN, SB_XMAX)
    ax.set_ylim(SB_YMIN, SB_YMAX)  # Remove the -0.3 arbitrary extension
    ax.set_aspect('equal')
    
    # Styling
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#cccccc')
        spine.set_linewidth(1.0)
    
    # Legend
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0, 
             markerscale=2.5, frameon=True, fancybox=True, framealpha=0.9)
    
    plt.savefig(output_dir / filename, dpi=300, bbox_inches="tight")
    print(f"Saved: {filename}")
    plt.show()
if 'nwis_sites_gdf' in locals() and 'historical_sites_gdf' in locals():
    create_comparison_map(nwis_sites_gdf, historical_sites_gdf, "Sites", "streamflow_sites_comparison.png")

# ==== Map 2: Measurement-Based Comparison Map ====
print("\n=== CREATING MEASUREMENT-BASED COMPARISON MAP ===")

if 'nwis_measurements_gdf' in locals() and 'historical_sites_gdf' in locals():
    create_comparison_map(nwis_measurements_gdf, historical_sites_gdf, "Measurements", "streamflow_measurements_comparison.png")

# ==== Map 3: Site-Based Decadal Map ====
print("\n=== CREATING SITE-BASED DECADAL MAP ===")

def create_decadal_map(nwis_data, historical_data, title_suffix, filename):
    """Create a decadal map"""
    # Combine datasets
    nwis_subset = nwis_data[['geometry', 'decade', 'data_type']].copy()
    historical_subset = historical_data[['geometry', 'decade', 'data_type']].copy()
    all_data = gpd.GeoDataFrame(pd.concat([nwis_subset, historical_subset], ignore_index=True))
    
    print(f"Combined {title_suffix.lower()} data: {len(all_data):,}")
    
    # Define decades
    decades = list(range(1860, 1980, 10))
    
    # Set up plot
    fig, axes = plt.subplots(3, 4, figsize=(12, 8), dpi=300)
    axes = axes.flatten()
    
    plt.subplots_adjust(left=0.05, bottom=0.05, right=0.85, top=0.95, wspace=0.1, hspace=0.15)
    
    colors = {'Restored USGS': '#EE6677', 'NWIS USGS': '#4477AA'}
    
    
    for i, decade in enumerate(decades):
        ax = axes[i]
        decade_data = all_data[all_data['decade'] == decade]
        
        # Add California context
        try:
            ca_context_path = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\ca_shapefile\ca_poly.shp"
            if os.path.exists(ca_context_path):
                ca_context = gpd.read_file(ca_context_path).to_crs("EPSG:4326")
                ca_context_geom = ca_context.union_all() if hasattr(ca_context, "union_all") else ca_context.unary_union
                gpd.GeoSeries([ca_context_geom], crs="EPSG:4326").plot(
                    ax=ax, color="#f0f0f0", edgecolor="#e0e0e0", linewidth=0.3, zorder=0
                )
        except Exception:
            pass
        
        # Plot county boundary
        if SB_COUNTY_GDF is not None:
            SB_COUNTY_GDF.plot(ax=ax, color="lightgray", edgecolor="black", linewidth=0.5, zorder=1)
        
        # Plot data by type
        for data_type, color in [('NWIS USGS', colors['NWIS USGS']), ('Restored USGS', colors['Restored USGS'])]:
            type_data = decade_data[decade_data['data_type'] == data_type]
            if len(type_data) > 0:
                type_data.plot(ax=ax, color=color, markersize=10, alpha=0.8, edgecolor='none', 
                             linewidth=0, zorder=3 if data_type == 'NWIS USGS' else 2)
        
        # CONSISTENT extent setting (same for all panels)
        ax.set_xlim(SB_XMIN, SB_XMAX)
        ax.set_ylim(SB_YMIN, SB_YMAX)
        ax.set_aspect('equal')
        
        total_count = len(decade_data)
        ax.set_title(f"{decade}s ({total_count} {title_suffix.lower()})", fontsize=10, pad=5)
        
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color('#dddddd')
            spine.set_linewidth(0.5)
    
    # Legend
    legend_elements = [
        Line2D([0],[0], marker='o', linestyle='', markerfacecolor=colors['NWIS USGS'], 
               markeredgecolor='none', markersize=8, label='NWIS USGS Stream Data'),
        Line2D([0],[0], marker='o', linestyle='', markerfacecolor=colors['Restored USGS'], 
               markeredgecolor='none', markersize=8, label='Restored USGS Stream Data')
    ]
    
    fig.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(0.87, 0.5),
               frameon=True, fancybox=True, framealpha=0.9, fontsize=9)
    
    plt.savefig(output_dir / filename, dpi=300, bbox_inches="tight")
    print(f"Saved {title_suffix.lower()} decadal map: {filename}")
    plt.show()
    
    # Print summary
    print(f"\n=== {title_suffix.upper()} DECADAL SUMMARY ===")
    decade_summary = all_data.groupby(['decade', 'data_type']).size().unstack(fill_value=0)
    print(f"{title_suffix} by decade and data type:")
    print(decade_summary)

if 'nwis_sites_gdf' in locals() and 'historical_sites_gdf' in locals():
    create_decadal_map(nwis_sites_gdf, historical_sites_gdf, "Sites", "streamflow_sites_decadal.png")

# ==== Map 4: Measurement-Based Decadal Map ====
print("\n=== CREATING MEASUREMENT-BASED DECADAL MAP ===")

if 'nwis_measurements_gdf' in locals() and 'historical_sites_gdf' in locals():
    create_decadal_map(nwis_measurements_gdf, historical_sites_gdf, "Measurements", "streamflow_measurements_decadal.png")

print("\n=== ALL MAPS COMPLETED ===")
print("Created 4 streamflow maps:")
print("1. Site-based comparison")
print("2. Measurement-based comparison") 
print("3. Site-based decadal")
print("4. Measurement-based decadal")

In [ ]:
# Add this to your existing code after loading the data
import folium
from folium import plugins
import json

print("\n=== CREATING INTERACTIVE MAP ===")

def create_interactive_streamflow_map_with_decades():
    """Create an interactive map with decade filtering"""
    
    # Calculate map center from your data
    all_lats = []
    all_lons = []
    
    # Use the variables from your existing script
    if len(nwis_sites_gdf) > 0:
        all_lats.extend(nwis_sites_gdf['latitude'].dropna().tolist())
        all_lons.extend(nwis_sites_gdf['longitude'].dropna().tolist())
    
    if len(historical_sites_gdf) > 0:
        all_lats.extend(historical_sites_gdf['lat_combined'].dropna().tolist())
        all_lons.extend(historical_sites_gdf['lon_combined'].dropna().tolist())
    
    center_lat = sum(all_lats) / len(all_lats) if all_lats else 34.5
    center_lon = sum(all_lons) / len(all_lons) if all_lons else -120.0
    
    # Create base map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=9,
        tiles='OpenStreetMap'
    )
    
    # Add alternative tile layers
    folium.TileLayer(
        tiles='https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png',
        attr='© OpenTopoMap contributors',
        name='Topographic'
    ).add_to(m)
    folium.TileLayer(
        tiles='CartoDB positron',
        attr='© CartoDB',
        name='Light Theme'
    ).add_to(m)
    
    # Get all unique decades from both datasets
    all_decades = set()
    
    # Process NWIS data to add decades if not present
    nwis_with_decades = nwis_sites_gdf.copy()
    if 'decade' not in nwis_with_decades.columns and 'begin_date' in nwis_with_decades.columns:
        nwis_with_decades['begin_year'] = pd.to_datetime(nwis_with_decades['begin_date'], errors='coerce').dt.year
        nwis_with_decades['decade'] = (nwis_with_decades['begin_year'] // 10) * 10
    
    # Process historical data to ensure decades
    historical_with_decades = historical_sites_gdf.copy()
    if 'decade' not in historical_with_decades.columns and 'start_year' in historical_with_decades.columns:
        historical_with_decades['start_year_num'] = pd.to_numeric(historical_with_decades['start_year'], errors='coerce')
        historical_with_decades['decade'] = (historical_with_decades['start_year_num'] // 10) * 10
    
    # Collect all decades
    if 'decade' in nwis_with_decades.columns:
        nwis_decades = nwis_with_decades['decade'].dropna().unique()
        all_decades.update(nwis_decades)
    
    if 'decade' in historical_with_decades.columns:
        hist_decades = historical_with_decades['decade'].dropna().unique()
        all_decades.update(hist_decades)
    
    # Sort decades
    sorted_decades = sorted([int(d) for d in all_decades if pd.notna(d)])
    
    # Create "All Data" groups
    all_nwis_group = folium.FeatureGroup(name='All NWIS USGS Sites')
    all_historical_group = folium.FeatureGroup(name='All Restored USGS Sites')
    
    # Create decade-specific groups
    decade_groups = {}
    for decade in sorted_decades:
        decade_groups[decade] = {
            'nwis': folium.FeatureGroup(name=f'NWIS {decade}s'),
            'historical': folium.FeatureGroup(name=f'Restored {decade}s')
        }
    
    # Add NWIS sites
    if len(nwis_with_decades) > 0:
        for idx, row in nwis_with_decades.iterrows():
            if pd.notna(row['latitude']) and pd.notna(row['longitude']):
                
                # Create popup with original data
                popup_html = f"""
                <div style="width: 300px;">
                    <h4>NWIS USGS Site</h4>
                    <p><strong>Site Number:</strong> {row.get('site_no', 'N/A')}</p>
                    <p><strong>Station Name:</strong> {row.get('station_nm', 'N/A')}</p>
                    <p><strong>Begin Date:</strong> {row.get('begin_date', 'N/A')}</p>
                    <p><strong>End Date:</strong> {row.get('end_date', 'N/A')}</p>
                    <p><strong>Decade:</strong> {row.get('decade', 'Unknown')}s</p>
                    
                    <hr>
                    <h5>Source Data:</h5>
                    <p><strong>CSV Row Index:</strong> {idx}</p>
                    <p><strong>Source File:</strong> sb_streamflow_sites_for_mapping.csv</p>
                    
                    <details>
                        <summary>View All Original Data</summary>
                        <table style="font-size: 10px; width: 100%;">
                """
                
                # Add all original data fields
                for col, val in row.items():
                    if pd.notna(val):
                        popup_html += f"<tr><td><strong>{col}:</strong></td><td>{val}</td></tr>"
                
                popup_html += """
                        </table>
                    </details>
                </div>
                """
                
                marker = folium.CircleMarker(
                    location=[row['latitude'], row['longitude']],
                    radius=8,
                    popup=folium.Popup(popup_html, max_width=400),
                    color='#4477AA',
                    fill=True,
                    fillColor='#4477AA',
                    fillOpacity=0.7,
                    weight=2,
                    tooltip=f"NWIS Site {row.get('site_no', 'Unknown')}"
                )
                
                # Add to all data group
                marker.add_to(all_nwis_group)
                
                # Add to decade-specific group if decade is available
                if pd.notna(row.get('decade')) and int(row['decade']) in decade_groups:
                    decade_marker = folium.CircleMarker(
                        location=[row['latitude'], row['longitude']],
                        radius=8,
                        popup=folium.Popup(popup_html, max_width=400),
                        color='#4477AA',
                        fill=True,
                        fillColor='#4477AA',
                        fillOpacity=0.7,
                        weight=2,
                        tooltip=f"NWIS Site {row.get('site_no', 'Unknown')} ({int(row['decade'])}s)"
                    )
                    decade_marker.add_to(decade_groups[int(row['decade'])]['nwis'])
    
    # Add historical sites
    if len(historical_with_decades) > 0:
        for idx, row in historical_with_decades.iterrows():
            if pd.notna(row['lat_combined']) and pd.notna(row['lon_combined']):
                
                popup_html = f"""
                <div style="width: 300px;">
                    <h4>Restored USGS Site</h4>
                    <p><strong>Station Name:</strong> {row.get('station_name', 'N/A')}</p>
                    <p><strong>Years:</strong> {row.get('start_year', 'N/A')} - {row.get('end_year', 'N/A')}</p>
                    <p><strong>Water Type:</strong> {row.get('water_type_clean', 'N/A')}</p>
                    <p><strong>Decade:</strong> {row.get('decade', 'Unknown')}s</p>
                    
                    <hr>
                    <h5>Source Data:</h5>
                    <p><strong>CSV Row Index:</strong> {idx}</p>
                    <p><strong>Source File:</strong> santa_barbara_sites.csv</p>
                    
                    <details>
                        <summary>View All Original Data</summary>
                        <table style="font-size: 10px;">
                """
                
                for col, val in row.items():
                    if pd.notna(val):
                        popup_html += f"<tr><td><strong>{col}:</strong></td><td>{val}</td></tr>"
                
                popup_html += """
                        </table>
                    </details>
                </div>
                """
                
                marker = folium.CircleMarker(
                    location=[row['lat_combined'], row['lon_combined']],
                    radius=8,
                    popup=folium.Popup(popup_html, max_width=400),
                    color='#EE6677',
                    fill=True,
                    fillColor='#EE6677',
                    fillOpacity=0.7,
                    weight=2,
                    tooltip=f"Historical: {row.get('station_name', 'Unknown')}"
                )
                
                # Add to all data group
                marker.add_to(all_historical_group)
                
                # Add to decade-specific group
                if pd.notna(row.get('decade')) and int(row['decade']) in decade_groups:
                    decade_marker = folium.CircleMarker(
                        location=[row['lat_combined'], row['lon_combined']],
                        radius=8,
                        popup=folium.Popup(popup_html, max_width=400),
                        color='#EE6677',
                        fill=True,
                        fillColor='#EE6677',
                        fillOpacity=0.7,
                        weight=2,
                        tooltip=f"Historical: {row.get('station_name', 'Unknown')} ({int(row['decade'])}s)"
                    )
                    decade_marker.add_to(decade_groups[int(row['decade'])]['historical'])
    
    # Add all groups to map
    all_nwis_group.add_to(m)
    all_historical_group.add_to(m)
    
    # Add decade groups to map
    for decade in sorted_decades:
        decade_groups[decade]['nwis'].add_to(m)
        decade_groups[decade]['historical'].add_to(m)
    
    # Add layer control with better visibility
    layer_control = folium.LayerControl(
        position='topleft', 
        collapsed=False,  # Keep it expanded
        autoZIndex=True
    )
    layer_control.add_to(m)
    
    print(f"Added layer control with {len(sorted_decades)} decades")
    print("Decade groups created:")
    for decade in sorted_decades:
        nwis_count = len(nwis_with_decades[nwis_with_decades.get('decade', 0) == decade]) if 'decade' in nwis_with_decades.columns else 0
        hist_count = len(historical_with_decades[historical_with_decades.get('decade', 0) == decade]) if 'decade' in historical_with_decades.columns else 0
        print(f"  {decade}s: {nwis_count} NWIS sites, {hist_count} historical sites")
    
    # Add county boundary if available
    if 'sb_county' in globals():
        folium.GeoJson(
            sb_county.to_json(),
            style_function=lambda feature: {
                'fillColor': 'none',
                'color': 'black',
                'weight': 2,
                'fillOpacity': 0
            }
        ).add_to(m)
    
    # Add decade summary information with troubleshooting
    decades_found = len(sorted_decades)
    nwis_total = len(nwis_with_decades) if len(nwis_with_decades) > 0 else 0
    hist_total = len(historical_with_decades) if len(historical_with_decades) > 0 else 0
    
    decade_info = f"""
    <div style="
        position: fixed;
        top: 10px;
        right: 10px;
        background: rgba(255,255,255,0.95);
        padding: 15px;
        border-radius: 5px;
        box-shadow: 0 2px 10px rgba(0,0,0,0.3);
        z-index: 1000;
        font-family: Arial, sans-serif;
        font-size: 12px;
        max-width: 250px;
        border: 2px solid #007cba;
    ">
        <h4 style="margin: 0 0 10px 0; color: #007cba;">Streamflow Data Summary</h4>
        <p style="margin: 0 0 5px 0;"><strong>Total Sites:</strong> {nwis_total + hist_total}</p>
        <p style="margin: 0 0 5px 0;"><strong>NWIS Sites:</strong> {nwis_total}</p>
        <p style="margin: 0 0 10px 0;"><strong>Historical Sites:</strong> {hist_total}</p>
        
        <p style="margin: 0 0 5px 0;"><strong>Decades Found:</strong> {decades_found}</p>
        <p style="margin: 0 0 10px 0; font-size: 11px;">{', '.join([f"{d}s" for d in sorted_decades]) if sorted_decades else 'None detected'}</p>
        
        <div style="background: #f0f8ff; padding: 8px; border-radius: 3px; border-left: 3px solid #007cba;">
            <strong>🔍 Layer Control:</strong><br>
            Look for a box in the <strong>top-left corner</strong> with checkboxes to filter by decade and data type.
        </div>
        
        <p style="margin: 10px 0 0 0; font-size: 10px; color: #666;">
        Click any point to see source CSV data
        </p>
    </div>
    """
    
    # Add custom decade filter as backup
    custom_filter_html = f"""
    <style>
        .decade-filter {{
            position: fixed;
            top: 80px;
            left: 10px;
            background: white;
            padding: 15px;
            border-radius: 5px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.3);
            z-index: 1001;
            font-family: Arial, sans-serif;
            border: 2px solid #28a745;
            min-width: 200px;
        }}
        .decade-filter h4 {{
            margin: 0 0 10px 0;
            color: #28a745;
            font-size: 14px;
        }}
        .decade-filter select {{
            width: 100%;
            padding: 8px;
            border: 1px solid #ccc;
            border-radius: 4px;
            font-size: 13px;
        }}
        .filter-info {{
            margin-top: 10px;
            font-size: 11px;
            color: #666;
            background: #f8f9fa;
            padding: 8px;
            border-radius: 3px;
        }}
    </style>
    
    <div class="decade-filter">
        <h4>🗓️ Decade Filter</h4>
        <p style="margin: 0 0 8px 0; font-size: 12px;">Select decade to show:</p>
        <select id="decade-dropdown" onchange="updateDecadeFilter()">
            <option value="all">All Decades</option>
            {"".join([f'<option value="{decade}">{decade}s</option>' for decade in sorted_decades])}
        </select>
        
        <div class="filter-info">
            <div id="current-filter">Showing: All decades</div>
            <div style="margin-top: 5px;">
                <strong>Available:</strong> {', '.join([f"{d}s" for d in sorted_decades])}
            </div>
        </div>
        
        <div style="margin-top: 10px; font-size: 11px; background: #e3f2fd; padding: 5px; border-radius: 3px;">
            💡 <strong>Tip:</strong> Also check the layer control box in the top-left corner for more options!
        </div>
    </div>
    
    <script>
        // Simple decade filtering function
        function updateDecadeFilter() {{
            var selectedDecade = document.getElementById('decade-dropdown').value;
            var statusDiv = document.getElementById('current-filter');
            
            if (selectedDecade === 'all') {{
                statusDiv.innerHTML = 'Showing: All decades';
            }} else {{
                statusDiv.innerHTML = 'Showing: ' + selectedDecade + 's only';
            }}
            
            // Note: This is a visual indicator. Use the layer control for actual filtering.
            console.log('Decade filter changed to:', selectedDecade);
        }}
    </script>
    """
    
    m.get_root().html.add_child(folium.Element(custom_filter_html))
    
    # Save the map
    output_path = output_dir / "interactive_streamflow_map_with_decades.html"
    m.save(str(output_path))
    print(f"Interactive map with decade filtering saved to: {output_path}")
    print(f"Available decades: {', '.join([f'{d}s' for d in sorted_decades])}")
    
    return m

def create_interactive_streamflow_map():
    """Create an interactive map with clickable points showing original data"""
    
    # Calculate map center from your data
    all_lats = []
    all_lons = []
    
    # Use the variables from your existing script
    if len(nwis_sites_gdf) > 0:
        all_lats.extend(nwis_sites_gdf['latitude'].dropna().tolist())
        all_lons.extend(nwis_sites_gdf['longitude'].dropna().tolist())
    
    if len(historical_sites_gdf) > 0:
        all_lats.extend(historical_sites_gdf['lat_combined'].dropna().tolist())
        all_lons.extend(historical_sites_gdf['lon_combined'].dropna().tolist())
    
    center_lat = sum(all_lats) / len(all_lats) if all_lats else 34.5
    center_lon = sum(all_lons) / len(all_lons) if all_lons else -120.0
    
    # Create base map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=9,
        tiles='OpenStreetMap'
    )
    
    # Add alternative tile layers
    folium.TileLayer(
        tiles='https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png',
        attr='© OpenTopoMap contributors',
        name='Topographic'
    ).add_to(m)
    folium.TileLayer(
        tiles='CartoDB positron',
        attr='© CartoDB',
        name='Light Theme'
    ).add_to(m)
    
    # Create feature groups for different data types
    nwis_group = folium.FeatureGroup(name='NWIS USGS Stream Data')
    historical_group = folium.FeatureGroup(name='Restored USGS Stream Data')
    
    # Add NWIS sites/measurements
    if len(nwis_sites_gdf) > 0:
        for idx, row in nwis_sites_gdf.iterrows():
            if pd.notna(row['latitude']) and pd.notna(row['longitude']):
                
                # Create popup with original data
                popup_html = f"""
                <div style="width: 300px;">
                    <h4>NWIS USGS Site</h4>
                    <p><strong>Site Number:</strong> {row.get('site_no', 'N/A')}</p>
                    <p><strong>Station Name:</strong> {row.get('station_nm', 'N/A')}</p>
                    <p><strong>Begin Date:</strong> {row.get('begin_date', 'N/A')}</p>
                    <p><strong>End Date:</strong> {row.get('end_date', 'N/A')}</p>
                    <p><strong>Coordinates:</strong> {row['latitude']:.6f}, {row['longitude']:.6f}</p>
                    
                    <hr>
                    <h5>Source Data:</h5>
                    <p><strong>CSV Row Index:</strong> {idx}</p>
                    <p><strong>Source File:</strong> sb_streamflow_sites_for_mapping.csv</p>
                    
                    <details>
                        <summary>View All Original Data</summary>
                        <table style="font-size: 10px; width: 100%;">
                """
                
                # Add all original data fields
                for col, val in row.items():
                    if pd.notna(val):
                        popup_html += f"<tr><td><strong>{col}:</strong></td><td>{val}</td></tr>"
                
                popup_html += """
                        </table>
                    </details>
                </div>
                """
                
                folium.CircleMarker(
                    location=[row['latitude'], row['longitude']],
                    radius=8,
                    popup=folium.Popup(popup_html, max_width=400),
                    color='#4477AA',
                    fill=True,
                    fillColor='#4477AA',
                    fillOpacity=0.7,
                    weight=2,
                    tooltip=f"NWIS Site {row.get('site_no', 'Unknown')}"
                ).add_to(nwis_group)
    
    # Add measurements if you want them instead of/in addition to sites
    if len(nwis_measurements_gdf) > 0:
        # Limit to first 1000 measurements to avoid overwhelming the map
        sample_measurements = nwis_measurements_gdf.sample(n=min(1000, len(nwis_measurements_gdf)))
        
        for idx, row in sample_measurements.iterrows():
            if pd.notna(row['latitude']) and pd.notna(row['longitude']):
                
                popup_html = f"""
                <div style="width: 300px;">
                    <h4>NWIS Measurement</h4>
                    <p><strong>Site Number:</strong> {row.get('site_no', 'N/A')}</p>
                    <p><strong>Measurement Date:</strong> {row.get('measurement_dt', 'N/A')}</p>
                    <p><strong>Discharge (cfs):</strong> {row.get('discharge_va', 'N/A')}</p>
                    <p><strong>Gage Height (ft):</strong> {row.get('gage_height_va', 'N/A')}</p>
                    
                    <hr>
                    <h5>Source Data:</h5>
                    <p><strong>CSV Row Index:</strong> {idx}</p>
                    <p><strong>Source File:</strong> sb_streamflow_usgs_1800_1979.csv</p>
                    
                    <details>
                        <summary>View All Original Data</summary>
                        <table style="font-size: 10px;">
                """
                
                for col, val in row.items():
                    if pd.notna(val):
                        popup_html += f"<tr><td><strong>{col}:</strong></td><td>{val}</td></tr>"
                
                popup_html += """
                        </table>
                    </details>
                </div>
                """
                
                folium.CircleMarker(
                    location=[row['latitude'], row['longitude']],
                    radius=6,
                    popup=folium.Popup(popup_html, max_width=400),
                    color='#4477AA',
                    fill=True,
                    fillColor='#4477AA',
                    fillOpacity=0.7,
                    weight=1,
                    tooltip=f"Measurement {row.get('measurement_dt', 'Unknown date')}"
                ).add_to(nwis_group)
    
    # Add historical sites
    if len(historical_sites_gdf) > 0:
        for idx, row in historical_sites_gdf.iterrows():
            if pd.notna(row['lat_combined']) and pd.notna(row['lon_combined']):
                
                popup_html = f"""
                <div style="width: 300px;">
                    <h4>Restored USGS Site</h4>
                    <p><strong>Station Name:</strong> {row.get('station_name', 'N/A')}</p>
                    <p><strong>Years:</strong> {row.get('start_year', 'N/A')} - {row.get('end_year', 'N/A')}</p>
                    <p><strong>Water Type:</strong> {row.get('water_type_clean', 'N/A')}</p>
                    <p><strong>Location:</strong> {row.get('location_description', 'N/A')}</p>
                    
                    <hr>
                    <h5>Source Data:</h5>
                    <p><strong>CSV Row Index:</strong> {idx}</p>
                    <p><strong>Source File:</strong> santa_barbara_sites.csv</p>
                    
                    <details>
                        <summary>View All Original Data</summary>
                        <table style="font-size: 10px;">
                """
                
                for col, val in row.items():
                    if pd.notna(val):
                        popup_html += f"<tr><td><strong>{col}:</strong></td><td>{val}</td></tr>"
                
                popup_html += """
                        </table>
                    </details>
                </div>
                """
                
                folium.CircleMarker(
                    location=[row['lat_combined'], row['lon_combined']],
                    radius=8,
                    popup=folium.Popup(popup_html, max_width=400),
                    color='#EE6677',
                    fill=True,
                    fillColor='#EE6677',
                    fillOpacity=0.7,
                    weight=2,
                    tooltip=f"Historical: {row.get('station_name', 'Unknown')}"
                ).add_to(historical_group)
    
    # Add feature groups to map
    nwis_group.add_to(m)
    historical_group.add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # Add decade filter control
    add_decade_filter_control(m, nwis_sites_gdf, historical_sites_gdf, nwis_group, historical_group)
    
    # Add a marker cluster plugin for better performance with lots of points
    # Uncomment these lines if you have many points:
    # marker_cluster = plugins.MarkerCluster().add_to(m)
    
    # Add county boundary if available
    if 'sb_county' in globals():
        folium.GeoJson(
            sb_county.to_json(),
            style_function=lambda feature: {
                'fillColor': 'none',
                'color': 'black',
                'weight': 2,
                'fillOpacity': 0
            }
        ).add_to(m)
    
    # Save the map
    output_path = output_dir / "interactive_streamflow_map.html"
    m.save(str(output_path))
    print(f"Interactive map saved to: {output_path}")
    
    # Display in Jupyter if available
    try:
        return m
    except:
        pass
    
    return m

# Create the interactive map with decade filtering
interactive_map_with_decades = create_interactive_streamflow_map_with_decades()

# You can still create the original version too if you want:
# interactive_map = create_interactive_streamflow_map()
def create_plotly_interactive_map():
    """Create a Plotly interactive map with advanced features"""
    
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    
    fig = go.Figure()
    
    # Add NWIS sites
    if len(nwis_sites_gdf) > 0:
        hover_text = []
        for idx, row in nwis_sites_gdf.iterrows():
            hover_info = f"""
            <b>NWIS USGS Site</b><br>
            Site: {row.get('site_no', 'N/A')}<br>
            Name: {row.get('station_nm', 'N/A')}<br>
            Begin: {row.get('begin_date', 'N/A')}<br>
            End: {row.get('end_date', 'N/A')}<br>
            Row Index: {idx}<br>
            Source: sb_streamflow_sites_for_mapping.csv
            """
            hover_text.append(hover_info)
        
        fig.add_trace(go.Scattermapbox(
            lat=nwis_sites_gdf['latitude'],
            lon=nwis_sites_gdf['longitude'],
            mode='markers',
            marker=dict(size=8, color='#4477AA'),
            name='NWIS USGS',
            hovertemplate='%{text}<extra></extra>',
            text=hover_text
        ))
    
    # Add historical sites
    if len(historical_sites_gdf) > 0:
        hover_text_hist = []
        for idx, row in historical_sites_gdf.iterrows():
            hover_info = f"""
            <b>Restored USGS Site</b><br>
            Station: {row.get('station_name', 'N/A')}<br>
            Years: {row.get('start_year', 'N/A')}-{row.get('end_year', 'N/A')}<br>
            Type: {row.get('water_type_clean', 'N/A')}<br>
            Row Index: {idx}<br>
            Source: santa_barbara_sites.csv
            """
            hover_text_hist.append(hover_info)
        
        fig.add_trace(go.Scattermapbox(
            lat=historical_sites_gdf['lat_combined'],
            lon=historical_sites_gdf['lon_combined'],
            mode='markers',
            marker=dict(size=8, color='#EE6677'),
            name='Restored USGS',
            hovertemplate='%{text}<extra></extra>',
            text=hover_text_hist
        ))
    
    # Update layout
    fig.update_layout(
        mapbox=dict(
            accesstoken='pk.eyJ1IjoicGxvdGx5bWFwYm94IiwiYSI6ImNrOWJqb2F4djBnMjEzbG50amg0dnJieG4ifQ.Zme1-Oumv-CYqLxVkRWHLw',
            style='open-street-map',
            center=dict(lat=center_lat, lon=center_lon),
            zoom=8
        ),
        title='Interactive Streamflow Data Map',
        height=600
    )
    
    # Save as HTML
    plotly_path = output_dir / "plotly_streamflow_map.html"
    fig.write_html(str(plotly_path))
    print(f"Plotly map saved to: {plotly_path}")
    
    return fig

# Uncomment to create Plotly version:
# plotly_map = create_plotly_interactive_map()

In [ ]:
def create_simple_interactive_streamflow_map():
    """Create an interactive map with working decade filtering - simplified approach"""
    
    # Calculate map center from your data
    all_lats = []
    all_lons = []
    
    if len(nwis_sites_gdf) > 0:
        all_lats.extend(nwis_sites_gdf['latitude'].dropna().tolist())
        all_lons.extend(nwis_sites_gdf['longitude'].dropna().tolist())
    
    if len(historical_sites_gdf) > 0:
        all_lats.extend(historical_sites_gdf['lat_combined'].dropna().tolist())
        all_lons.extend(historical_sites_gdf['lon_combined'].dropna().tolist())
    
    center_lat = sum(all_lats) / len(all_lats) if all_lats else 34.5
    center_lon = sum(all_lons) / len(all_lons) if all_lons else -120.0
    
    # Create base map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=9,
        tiles='OpenStreetMap'
    )
    
    # Add alternative tile layers
    folium.TileLayer(
        tiles='https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png',
        attr='© OpenTopoMap contributors',
        name='Topographic'
    ).add_to(m)
    
    # Prepare data with decades
    nwis_with_decades = nwis_sites_gdf.copy()
    if 'decade' not in nwis_with_decades.columns and 'begin_date' in nwis_with_decades.columns:
        nwis_with_decades['begin_year'] = pd.to_datetime(nwis_with_decades['begin_date'], errors='coerce').dt.year
        nwis_with_decades['decade'] = (nwis_with_decades['begin_year'] // 10) * 10
    
    historical_with_decades = historical_sites_gdf.copy()
    if 'decade' not in historical_with_decades.columns and 'start_year' in historical_with_decades.columns:
        historical_with_decades['start_year_num'] = pd.to_numeric(historical_with_decades['start_year'], errors='coerce')
        historical_with_decades['decade'] = (historical_with_decades['start_year_num'] // 10) * 10
    
    # Get all unique decades
    all_decades = set()
    if 'decade' in nwis_with_decades.columns:
        all_decades.update(nwis_with_decades['decade'].dropna().unique())
    if 'decade' in historical_with_decades.columns:
        all_decades.update(historical_with_decades['decade'].dropna().unique())
    
    sorted_decades = sorted([int(d) for d in all_decades if pd.notna(d)])
    
    # Create all markers data for JavaScript
    markers_data = []
    
    # Add NWIS sites
    if len(nwis_with_decades) > 0:
        for idx, row in nwis_with_decades.iterrows():
            if pd.notna(row['latitude']) and pd.notna(row['longitude']):
                popup_html = f"""
                <div style="width: 300px;">
                    <h4>NWIS USGS Site</h4>
                    <p><strong>Site Number:</strong> {row.get('site_no', 'N/A')}</p>
                    <p><strong>Station Name:</strong> {row.get('station_nm', 'N/A')}</p>
                    <p><strong>Begin Date:</strong> {row.get('begin_date', 'N/A')}</p>
                    <p><strong>End Date:</strong> {row.get('end_date', 'N/A')}</p>
                    <p><strong>Decade:</strong> {row.get('decade', 'Unknown')}s</p>
                    <hr>
                    <h5>Source Data:</h5>
                    <p><strong>CSV Row Index:</strong> {idx}</p>
                    <p><strong>Source File:</strong> sb_streamflow_sites_for_mapping.csv</p>
                    <details>
                        <summary>View All Original Data</summary>
                        <table style="font-size: 10px; width: 100%;">
                """
                
                for col, val in row.items():
                    if pd.notna(val):
                        popup_html += f"<tr><td><strong>{col}:</strong></td><td>{val}</td></tr>"
                
                popup_html += """
                        </table>
                    </details>
                </div>
                """
                
                marker_info = {
                    'lat': float(row['latitude']),
                    'lng': float(row['longitude']),
                    'decade': int(row.get('decade', 0)) if pd.notna(row.get('decade')) else 0,
                    'type': 'nwis',
                    'popup': popup_html,
                    'tooltip': f"NWIS Site {row.get('site_no', 'Unknown')}"
                }
                markers_data.append(marker_info)
    
    # Add historical sites
    if len(historical_with_decades) > 0:
        for idx, row in historical_with_decades.iterrows():
            if pd.notna(row['lat_combined']) and pd.notna(row['lon_combined']):
                popup_html = f"""
                <div style="width: 300px;">
                    <h4>Restored USGS Site</h4>
                    <p><strong>Station Name:</strong> {row.get('station_name', 'N/A')}</p>
                    <p><strong>Years:</strong> {row.get('start_year', 'N/A')} - {row.get('end_year', 'N/A')}</p>
                    <p><strong>Water Type:</strong> {row.get('water_type_clean', 'N/A')}</p>
                    <p><strong>Decade:</strong> {row.get('decade', 'Unknown')}s</p>
                    <hr>
                    <h5>Source Data:</h5>
                    <p><strong>CSV Row Index:</strong> {idx}</p>
                    <p><strong>Source File:</strong> santa_barbara_sites.csv</p>
                    <details>
                        <summary>View All Original Data</summary>
                        <table style="font-size: 10px;">
                """
                
                for col, val in row.items():
                    if pd.notna(val):
                        popup_html += f"<tr><td><strong>{col}:</strong></td><td>{val}</td></tr>"
                
                popup_html += """
                        </table>
                    </details>
                </div>
                """
                
                marker_info = {
                    'lat': float(row['lat_combined']),
                    'lng': float(row['lon_combined']),
                    'decade': int(row.get('decade', 0)) if pd.notna(row.get('decade')) else 0,
                    'type': 'historical',
                    'popup': popup_html,
                    'tooltip': f"Historical: {row.get('station_name', 'Unknown')}"
                }
                markers_data.append(marker_info)
    
    # Add county boundary if available
    if 'sb_county' in globals():
        folium.GeoJson(
            sb_county.to_json(),
            style_function=lambda feature: {
                'fillColor': 'none',
                'color': 'black',
                'weight': 2,
                'fillOpacity': 0
            }
        ).add_to(m)
    
    # Create the interactive filter with all markers controlled by JavaScript
    markers_json = json.dumps(markers_data)
    decades_json = json.dumps(sorted_decades)
    
    interactive_html = f"""
    <style>
        .filter-control {{
            position: fixed;
            top: 10px;
            left: 10px;
            background: white;
            padding: 15px;
            border-radius: 8px;
            box-shadow: 0 3px 15px rgba(0,0,0,0.3);
            z-index: 1000;
            font-family: Arial, sans-serif;
            border: 2px solid #28a745;
            min-width: 250px;
        }}
        .filter-control h3 {{
            margin: 0 0 15px 0;
            color: #28a745;
            font-size: 16px;
        }}
        .filter-group {{
            margin-bottom: 15px;
        }}
        .filter-group label {{
            display: block;
            margin-bottom: 5px;
            font-weight: bold;
            font-size: 13px;
        }}
        .filter-group select {{
            width: 100%;
            padding: 8px;
            border: 1px solid #ddd;
            border-radius: 4px;
            font-size: 13px;
        }}
        .filter-stats {{
            background: #f8f9fa;
            padding: 10px;
            border-radius: 4px;
            font-size: 12px;
            margin-top: 10px;
        }}
        .legend {{
            margin-top: 15px;
            padding: 10px;
            background: #e8f5e8;
            border-radius: 4px;
        }}
        .legend-item {{
            display: flex;
            align-items: center;
            margin-bottom: 5px;
            font-size: 12px;
        }}
        .legend-color {{
            width: 12px;
            height: 12px;
            border-radius: 50%;
            margin-right: 8px;
        }}
    </style>
    
    <div class="filter-control">
        <h3>🗺️ Streamflow Data Explorer</h3>
        
        <div class="filter-group">
            <label>Filter by Decade:</label>
            <select id="decade-filter" onchange="applyFilters()">
                <option value="all">All Decades</option>
                {"".join([f'<option value="{decade}">{decade}s</option>' for decade in sorted_decades])}
            </select>
        </div>
        
        <div class="filter-group">
            <label>Filter by Data Type:</label>
            <select id="type-filter" onchange="applyFilters()">
                <option value="all">All Types</option>
                <option value="nwis">NWIS USGS Only</option>
                <option value="historical">Restored USGS Only</option>
            </select>
        </div>
        
        <div class="filter-stats">
            <div id="filter-status">Loading data...</div>
            <div id="point-count" style="margin-top: 5px; font-weight: bold;"></div>
        </div>
        
        <div class="legend">
            <div class="legend-item">
                <div class="legend-color" style="background-color: #4477AA;"></div>
                <span>NWIS USGS Sites</span>
            </div>
            <div class="legend-item">
                <div class="legend-color" style="background-color: #EE6677;"></div>
                <span>Restored USGS Sites</span>
            </div>
        </div>
    </div>
    
    <script>
        // Global variables
        var allMarkers = {markers_json};
        var currentMarkers = [];
        var map = null;
        
        // Wait for map to be ready
        window.addEventListener('load', function() {{
            setTimeout(function() {{
                initializeMap();
            }}, 1000);
        }});
        
        function initializeMap() {{
            // Find the Leaflet map instance
            if (window[Object.keys(window).find(key => key.startsWith('map_'))]) {{
                map = window[Object.keys(window).find(key => key.startsWith('map_'))];
                console.log('Map found!', map);
                createAllMarkers();
            }} else {{
                console.log('Map not found, retrying...');
                setTimeout(initializeMap, 1000);
            }}
        }}
        
        function createAllMarkers() {{
            currentMarkers = [];
            
            allMarkers.forEach(function(markerData) {{
                var color = markerData.type === 'nwis' ? '#4477AA' : '#EE6677';
                
                var marker = L.circleMarker([markerData.lat, markerData.lng], {{
                    color: color,
                    fillColor: color,
                    fillOpacity: 0.7,
                    radius: 6,
                    weight: 2
                }});
                
                marker.bindPopup(markerData.popup, {{maxWidth: 400}});
                marker.bindTooltip(markerData.tooltip);
                
                // Store metadata
                marker._streamflowData = markerData;
                
                marker.addTo(map);
                currentMarkers.push(marker);
            }});
            
            updateStats();
        }}
        
        function applyFilters() {{
            var decadeFilter = document.getElementById('decade-filter').value;
            var typeFilter = document.getElementById('type-filter').value;
            
            var visibleCount = 0;
            
            currentMarkers.forEach(function(marker) {{
                var data = marker._streamflowData;
                var showMarker = true;
                
                // Apply decade filter
                if (decadeFilter !== 'all') {{
                    if (data.decade !== parseInt(decadeFilter)) {{
                        showMarker = false;
                    }}
                }}
                
                // Apply type filter
                if (typeFilter !== 'all') {{
                    if (data.type !== typeFilter) {{
                        showMarker = false;
                    }}
                }}
                
                if (showMarker) {{
                    marker.setStyle({{opacity: 1, fillOpacity: 0.7}});
                    visibleCount++;
                }} else {{
                    marker.setStyle({{opacity: 0, fillOpacity: 0}});
                }}
            }});
            
            updateStats(visibleCount, decadeFilter, typeFilter);
        }}
        
        function updateStats(visibleCount, decade, type) {{
            var statusDiv = document.getElementById('filter-status');
            var countDiv = document.getElementById('point-count');
            
            if (visibleCount === undefined) {{
                visibleCount = currentMarkers.length;
                decade = 'all';
                type = 'all';
            }}
            
            var statusText = 'Showing: ';
            if (decade === 'all' && type === 'all') {{
                statusText += 'All data';
            }} else {{
                if (decade !== 'all') statusText += decade + 's ';
                if (type !== 'all') statusText += (type === 'nwis' ? 'NWIS' : 'Restored') + ' ';
                statusText += 'data';
            }}
            
            statusDiv.innerHTML = statusText;
            countDiv.innerHTML = 'Visible points: ' + visibleCount + ' / ' + allMarkers.length;
        }}
    </script>
    """
    
    # Add the interactive HTML to the map
    m.get_root().html.add_child(folium.Element(interactive_html))
    
    # Save the map
    output_path = output_dir / "working_interactive_streamflow_map.html"
    m.save(str(output_path))
    print(f"Working interactive map saved to: {output_path}")
    print(f"Total markers: {len(markers_data)}")
    print(f"Available decades: {', '.join([f'{d}s' for d in sorted_decades])}")
    
    return m

# Create the working interactive map
print("\n=== CREATING WORKING INTERACTIVE MAP ===")
working_map = create_simple_interactive_streamflow_map()